In [1]:
from func import *

# Read data

In [2]:
train_df = pd.read_csv("processed5/train_data.csv")
test_df = pd.read_csv("processed5/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

# Elastic Net

In [3]:
from sklearn.linear_model import ElasticNet

In [3]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8627505332713833 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [4]:
alpha = 0.01778279410038923
l_1_ratio = 0.05
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': 0.8631442453920551, 'R2': 0.7519455718180945, 'MAPE': 16.6108856500107, 'MdAPE': 11.895740137262962, 'RMSE': 437058.2016757085, 'MAE': 200126.29117267486, 'Bias(mean residual)': -48150.484427286625, 'APE_95pct': 47.37217794293921, 'APE_99pct': 82.00411410733462, 'APE_max': 239.3487026238858, 'Train_R2(log)': 0.8992553084461341, 'Test_R2(log)': 0.8631442453920551, 'R2_gap': 0.03611106305407896}


# Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [6]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


In [3]:

rf_param_dist = {
    "model__n_estimators": randint(300, 1000),
    "model__max_depth": [None] + list(range(10, 60, 10)),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20)
}

model = make_model_pipeline(model=RandomForestRegressor(
    n_jobs=1,
    random_state=42),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)


rs = RandomizedSearchCV(
    model,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(rs.best_params_)

{'model__max_depth': 40, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 13, 'model__n_estimators': 713}


In [8]:
n_estimators = 713
max_depth = 40
max_features = 'log2'
min_samples_leaf = 8
min_samples_split = 13

rf_model = RandomForestRegressor(n_estimators=n_estimators,
                                 max_depth=max_depth,
                                 max_features=max_features,
                                 min_samples_leaf=min_samples_leaf,
                                 min_samples_split=min_samples_split)

rf_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=rf_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

rf_metrics = rf_result["Metrics"]
print(rf_metrics)

{'R2(log)': 0.7092030250258208, 'R2': 0.4826927607644168, 'MAPE': 23.871813799414575, 'MdAPE': 17.85266896789031, 'RMSE': 631160.8615229924, 'MAE': 298421.6249677806, 'Bias(mean residual)': -157959.70800648423, 'APE_95pct': 64.62811735126827, 'APE_99pct': 110.16973398248231, 'APE_max': 283.82579199393257, 'Train_R2(log)': 0.9695408677323267, 'Test_R2(log)': 0.7092030250258208, 'R2_gap': 0.26033784270650595}


# XGB

In [8]:
from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform

In [7]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}

model = make_model_numeric_only_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=1),
 num_cols=num_cols)

xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.7844598129752072, 'model__learning_rate': 0.05383345943137039, 'model__max_depth': 6, 'model__min_child_weight': 14, 'model__n_estimators': 1219, 'model__reg_alpha': 1.10973369349242, 'model__reg_lambda': 4.736558858366902, 'model__subsample': 0.755325405158244}


In [12]:
xgb_model = XGBRegressor(n_estimators=1219,
                         learning_rate=0.05383345943137039,
                         max_depth=6,
                         subsample=0.755325405158244,
                         colsample_bytree=0.7844598129752072,
                         min_child_weight=14,
                         reg_lambda=4.736558858366902,
                         reg_alpha=1.10973369349242)


xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.9224351015790591, 'R2': 0.8761122335205692, 'MAPE': 12.01222878909544, 'MdAPE': 8.254262976314786, 'RMSE': 308873.03389506624, 'MAE': 146114.01863160907, 'Bias(mean residual)': -8678.030874661754, 'APE_95pct': 35.62418455013493, 'APE_99pct': 61.826838475378985, 'APE_max': 196.5331785714288, 'Train_R2(log)': 0.953032013802761, 'Test_R2(log)': 0.9224351015790591, 'R2_gap': 0.030596912223701977}
